In [5]:
import numpy as np
import pandas as pd
import xarray as xr
from types import SimpleNamespace
from analysis_utils import pop_weight_sum

AGE_COHORTS = ["age0to4", "age5to64", "age65plus"]
regions = ["A", "B"]

pop0to4 = xr.DataArray([100, 50], coords={"region": regions}, dims="region")
pop5to64 = xr.DataArray([700, 850], coords={"region": regions}, dims="region")
pop65plus = xr.DataArray([200, 100], coords={"region": regions}, dims="region")
total_pop = pop0to4 + pop5to64 + pop65plus

socioeconomics = {
    "pop": total_pop,
    "pop0to4": pop0to4,
    "pop5to64": pop5to64,
    "pop65plus": pop65plus,
}

da = xr.DataArray(
    [[10.0, 20.0, 30.0], [15.0, 25.0, 35.0]],
    coords={"region": regions, "age_cohort": AGE_COHORTS},
    dims=["region", "age_cohort"],
)

def make_config(age_weight, rate, cohort):
    return SimpleNamespace(age_weight=age_weight, rate=rate, cohort=cohort, socioeconomics=socioeconomics)

In [6]:
config = make_config(age_weight=True, rate=False, cohort=None)
result = pop_weight_sum(da, config, impact=True)

cohort_pops = xr.concat([pop0to4, pop5to64, pop65plus], dim=pd.Index(AGE_COHORTS, name="age_cohort"))
expected = (da * cohort_pops / 100000).sum(dim="age_cohort")

print(result.name, "==", f"age_weighted_impact ->", result.name == "age_weighted_impact")
print(np.allclose(result, expected))
result

age_weighted_impact == age_weighted_impact -> True
True


<xarray.DataArray 'age_weighted_impact' (region: 2)> Size: 16B
array([0.21 , 0.255])
Coordinates:
  * region   (region) <U1 8B 'A' 'B'

In [7]:
config = make_config(age_weight=True, rate=True, cohort=None)
result = pop_weight_sum(da, config, impact=True)

pop_weight = cohort_pops / total_pop
expected = (da * pop_weight).sum(dim="age_cohort")

print(result.name == "age_weighted_impact")
print(np.allclose(result, expected))
result

True
True


<xarray.DataArray 'age_weighted_impact' (region: 2)> Size: 16B
array([21. , 25.5])
Coordinates:
  * region   (region) <U1 8B 'A' 'B'

In [8]:
cohort = "age5to64"
config = make_config(age_weight=False, rate=False, cohort=cohort)
result = pop_weight_sum(da, config, impact=True)

pop_col = f"pop{cohort[3:]}"
expected = da.sel(age_cohort=cohort) * socioeconomics[pop_col] / 100000

print(result.name == f"{cohort}_impact")
print(np.allclose(result, expected))
result

True
True


<xarray.DataArray 'age5to64_impact' (region: 2)> Size: 16B
array([0.14  , 0.2125])
Coordinates:
  * region      (region) <U1 8B 'A' 'B'
    age_cohort  <U9 36B 'age5to64'

In [9]:
config = make_config(age_weight=False, rate=True, cohort=cohort)
result = pop_weight_sum(da, config, impact=True)
expected = da.sel(age_cohort=cohort)

print(result.name == f"{cohort}_impact")
print(np.allclose(result, expected))
result

True
True


<xarray.DataArray 'age5to64_impact' (region: 2)> Size: 16B
array([20., 25.])
Coordinates:
  * region      (region) <U1 8B 'A' 'B'
    age_cohort  <U9 36B 'age5to64'